# NYC Elevator Complaints — One Building's Full Story

> **Narrative:** *"This isn't a one-time problem. Look at the pattern over three years."*

Use this to spotlight a single building in a council briefing, a press pitch, or as supporting evidence in Housing Court.  
Source: [NYC Open Data — DOB Elevator Complaint Dispositions](https://data.cityofnewyork.us/Housing-Development/DOB-Elevator-Complaint-Dispositions/kqwi-7ncn)

In [ ]:
# ── PARAMETERS — edit before presenting ──────────────────────────
# Option A: look up by address (SODA uppercase format)
HOUSE_NUMBER = "341"
HOUSE_STREET = "EAST 162 STREET"   # uppercase, no house number
COMMUNITY_BOARD = "203"            # borough digit + 2-digit CB

# Option B: look up by BIN (faster if you know it)
BIN = None   # e.g. "2129469"  — if set, overrides address lookup

# Other pilot buildings to try:
#   150 LEFFERTS AVENUE  / 309 (Brooklyn, D35 Hudson)
#   1150 TIFFANY STREET  / 202 (Bronx, D17 Sanchez)
#   2045 STORY AVENUE    / 209 (Bronx, D18 Farías)
#   509 WEST 155 STREET  / 112 (Manhattan, D10 De La Rosa)
#   33 SARATOGA AVENUE   / 316 (Brooklyn, D42 Banks)
# ─────────────────────────────────────────────────────────────────

import os, requests
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from datetime import datetime

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../../../.env"))
except ImportError:
    pass

TOKEN = os.getenv("SODA_APP_TOKEN")
if not TOKEN:
    print("⚠  SODA_APP_TOKEN not set")

SODA_URL = "https://data.cityofnewyork.us/resource/kqwi-7ncn.json"
BOROUGH  = {"1": "Manhattan", "2": "Bronx", "3": "Brooklyn", "4": "Queens", "5": "Staten Island"}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f5f5f5",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.family":      "sans-serif",
    "axes.titlesize":   14,
    "axes.labelsize":   11,
})

In [ ]:
def _get(params: dict) -> list:
    if TOKEN:
        params["$$app_token"] = TOKEN
    r = requests.get(SODA_URL, params=params, timeout=15)
    r.raise_for_status()
    return r.json()

if BIN:
    records = _get({"$where": f"bin='{BIN}' AND complaint_category IN ('6S','6M')",
                    "$order": "date_entered DESC", "$limit": "1000"})
    lookup = f"BIN {BIN}"
else:
    records = _get({"$where": f"house_number='{HOUSE_NUMBER}'"
                              f" AND house_street='{HOUSE_STREET}'"
                              f" AND community_board='{COMMUNITY_BOARD}'"
                              f" AND complaint_category IN ('6S','6M')",
                    "$order": "date_entered DESC", "$limit": "1000"})
    lookup = f"{HOUSE_NUMBER} {HOUSE_STREET}, CB{COMMUNITY_BOARD}"

print(f"Lookup: {lookup}")
print(f"Records found: {len(records)}")

if not records:
    print("\n⚠  No records. Check house number, street name (uppercase), and community board.")
else:
    rec0     = records[0]
    resolved = rec0.get("house_number", "") + " " + rec0.get("house_street", "")
    bin_id   = rec0.get("bin", "Unknown")
    cb       = rec0.get("community_board", "")
    borough  = BOROUGH.get(cb[0], "?") if cb else "?"
    print(f"\nBuilding: {resolved}, {borough}")
    print(f"BIN: {bin_id}  |  Community Board: CB{cb}")

## Year-Over-Year Summary

In [ ]:
if not records:
    print("No data.")
else:
    by_year: Counter = Counter()
    for r in records:
        date_str = r.get("date_entered", "")
        parts = date_str.split("/")
        year = parts[2] if len(parts) == 3 else "?"
        by_year[year] += 1

    years_sorted = sorted(by_year.keys())
    counts = [by_year[y] for y in years_sorted]

    colors = ["#e63946" if by_year[y] == max(by_year.values()) else "#457b9d" for y in years_sorted]

    fig, ax = plt.subplots(figsize=(8, 3.5))
    bars = ax.bar(years_sorted, counts, color=colors, edgecolor="white", width=0.55)

    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                str(count), ha="center", fontsize=11, fontweight="bold", color="#333")

    ax.set_ylabel("Complaints")
    ax.set_title(f"{resolved}, {borough} — Complaints by Year", pad=12)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    plt.show()

    print(f"Total on record: {len(records)}")
    recent = [r for r in records if r.get("date_entered", "").endswith(("2023", "2024", "2025"))]
    print(f"Last 3 years (2023–2025): {len(recent)} complaints")

## Complaint Timeline

Every dot is one complaint. The pattern — not just the count — is the story.

In [ ]:
if not records:
    print("No data.")
else:
    dates = []
    for r in records:
        date_str = r.get("date_entered", "")
        try:
            dates.append(datetime.strptime(date_str, "%m/%d/%Y"))
        except ValueError:
            pass
    dates.sort()

    fig, ax = plt.subplots(figsize=(11, 2.8))
    ax.scatter(dates, [1] * len(dates), s=80, color="#e63946", alpha=0.7, edgecolors="white", linewidth=0.5)
    ax.set_yticks([])
    ax.set_xlabel("Date")
    ax.set_title(f"{resolved}, {borough} — Complaint Timeline ({dates[0].year}–{dates[-1].year})", pad=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

    # Summer shading
    for year in range(dates[0].year, dates[-1].year + 1):
        ax.axvspan(datetime(year, 6, 1), datetime(year, 8, 31),
                   alpha=0.12, color="#e63946", label="Summer" if year == dates[0].year else "_")

    ax.legend(loc="upper left", fontsize=9)
    fig.tight_layout()
    plt.show()

## Full Complaint Log

In [ ]:
if records:
    import pandas as pd
    LABELS = {"6S": "Elevator complaint", "6M": "Elevator/escalator complaint"}
    rows = []
    for r in records:
        code = r.get("complaint_category", "?")
        rows.append({
            "Date":     r.get("date_entered", ""),
            "Code":     code,
            "Category": LABELS.get(code, code),
            "Status":   r.get("status", ""),
        })
    df = pd.DataFrame(rows)
    pd.set_option("display.max_rows", 100)
    display(df)